# 課程 02 - 探索 Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** 是用於建立 AI 代理的統一框架。它提供了一個簡潔且可組合的架構，包含四個核心組件：

- **Client** – 連接至 AI 模型端點並處理通信
- **Agent** – 用指令和工具定義包裝客戶端
- **Tools** – 使用模型可調用的自訂函式來擴展代理能力
- **Session** – 維持多回合互動的對話歷史

在本課程中，我們將使用這些概念構建一個 <strong>旅遊預訂代理</strong>，該代理可以檢查目的地的可用性。


## 設定


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## 理解 Agent 框架架構

Microsoft Agent 框架採用分層架構：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – 使用 `FoundryChatClient` 連接至 Azure OpenAI 部署。負責身份驗證、請求格式化與回應解析。
2. **Agent** – 透過 `provider.create_agent()` 由 client 建立，整合模型存取、指令（系統提示）與工具。
3. **Tools** – 用 `@tool` 裝飾的 Python 函式，agent 可呼叫以執行操作或取得資料。
4. **Session** – 使用 `agent.create_session()` 建立的 `AgentSession` 物件，儲存對話歷史，使 agent 能記住前文，支援多回合對話。

讓我們一步步構建每個層級。


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 使用 @tool 裝飾器新增工具

工具讓代理程式執行產生文字以外的動作。`@tool` 裝飾器會將一般的 Python 函式轉換為代理程式可以呼叫的項目。

重點：
- 使用 `Annotated[type, "description"]` 讓模型了解每個參數。
- 文件字串會變成模型看到的工具描述。
- `approval_mode="never_require"` 表示此工具會自動執行，無需使用者確認。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 建立一個帶工具的代理人

現在我們將客戶端、指示和工具結合成一個代理人。`instructions` 作為系統提示 — 它們定義了代理人的角色和行為。


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 使用 Session 進行多輪對話

`AgentSession`（透過 `agent.create_session()` 建立）會追蹤對話中的所有訊息。將相同的 session 傳遞給每次的 `agent.run()` 呼叫，代理人即可存取完整的對話歷史並參考先前的訊息。

我們傳入 `tools=[check_destination_availability]`，讓代理人在每輪對話中都能呼叫我們的可用性檢查器。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## 總結

在本課程中，你探討了 Microsoft 代理框架的四大支柱：

| 概念 | 你學到了什麼 |
|---------|------------------|
| <strong>客戶端</strong> | `FoundryChatClient` 使用基於憑證的身份驗證連接到 Azure OpenAI |
| <strong>代理</strong> | `provider.create_agent()` 將模型連接與指令及名稱捆綁在一起 |
| <strong>工具</strong> | `@tool` 裝飾器讓代理可以調用 Python 函式 |
| <strong>會話</strong> | `agent.create_session()` 維持多輪對話歷史 |

這些構建塊結合起來，創造能進行自然對話、調用外部函式並維持上下文的代理——為後續課程中更進階的代理模式奠定基礎。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
